# FERRUS ORBIS — orbis_main.ipynb
## Fregate 05 : Extraction decor Roblox → GLB

```
keyword OU asset_id
        ↓
   orbis_fetch.py  (HTTP : API Roblox + CDN textures)
        ↓
   metadata.json + textures/
        ↓
   orbis_core.py   (Blender headless : reconstruction + GLB)
        ↓
OUT/ decor_XXXX.glb  +  rapport_orbis.json
```

**POUR L'EMPEROR. POUR LA FLOTTE FERRUS.**

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 0 — GIT SYNC
# Monte le Drive + clone/pull le repo + sync le codebase
# A executer EN PREMIER a chaque session Colab
# ═══════════════════════════════════════════════════════════

import os, shutil, subprocess
from google.colab import drive

# ── 0.1 Monter Google Drive ──────────────────────────────
drive.mount('/content/drive', force_remount=False)

# ── 0.2 CONFIGURER ICI ───────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/FLOTTE-FERRUS'
GITHUB_REPO = 'https://github.com/kioka8877-ux/FLOTTE-FERRUS.git'
CLONE_DIR   = '/content/FLOTTE-FERRUS'

# ── 0.3 Clone ou Pull ────────────────────────────────────
if os.path.isdir(os.path.join(CLONE_DIR, '.git')):
    print('[GIT SYNC] Repo deja clone — git pull...')
    proc = subprocess.run(
        ['git', '-C', CLONE_DIR, 'pull', '--rebase'],
        capture_output=True, text=True
    )
    print(proc.stdout.strip() or 'Deja a jour.')
else:
    print('[GIT SYNC] Clone du repo...')
    proc = subprocess.run(
        ['git', 'clone', '--depth=1', GITHUB_REPO, CLONE_DIR],
        capture_output=True, text=True
    )
    print(proc.stdout.strip() or 'Clone OK.')
    if proc.returncode != 0:
        print('[GIT SYNC] ERREUR :', proc.stderr[-500:])
        raise RuntimeError('git clone echoue')

# ── 0.4 Sync le codebase sur Drive ───────────────────────
SRC_CODEBASE  = os.path.join(CLONE_DIR, '05_FERRUS_ORBIS', 'codebase')
DEST_CODEBASE = os.path.join(DRIVE_ROOT, '05_FERRUS_ORBIS', 'codebase')
os.makedirs(DEST_CODEBASE, exist_ok=True)

for fname in ['orbis_fetch.py', 'orbis_core.py']:
    src  = os.path.join(SRC_CODEBASE, fname)
    dest = os.path.join(DEST_CODEBASE, fname)
    if os.path.isfile(src):
        shutil.copy2(src, dest)
        print(f'[GIT SYNC] Copie : {fname} → Drive/05_FERRUS_ORBIS/codebase/')

# ── 0.5 Creer les dossiers requis sur Drive ──────────────
for folder in ['IN', 'OUT']:
    os.makedirs(os.path.join(DRIVE_ROOT, '05_FERRUS_ORBIS', folder), exist_ok=True)

# ── 0.6 Bilan ────────────────────────────────────────────
print()
print('[GIT SYNC] ══════════════════════════════════════')
print('[GIT SYNC] Codebase synchronise depuis GitHub')
print(f'[GIT SYNC] DRIVE_ROOT : {DRIVE_ROOT}')
for folder in ['IN', 'OUT', 'codebase']:
    path = os.path.join(DRIVE_ROOT, '05_FERRUS_ORBIS', folder)
    print(f'  {"OK" if os.path.isdir(path) else "MANQUANT"}  {path}')
print('[GIT SYNC] Passer a la cellule SETUP ▶')
print('[GIT SYNC] ══════════════════════════════════════')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 1 — SETUP
# Installe Blender 4.2 LTS + prepare les chemins
# Recycle locus_main.ipynb Cell 01
# ═══════════════════════════════════════════════════════════

import os, subprocess, shutil

# ── 1.1 Chemins ───────────────────────────────────────────
ORBIS_ROOT    = os.path.join(DRIVE_ROOT, '05_FERRUS_ORBIS')
IN_DIR        = os.path.join(ORBIS_ROOT, 'IN')
OUT_DIR       = os.path.join(ORBIS_ROOT, 'OUT')
FETCH_SCRIPT  = os.path.join(ORBIS_ROOT, 'codebase', 'orbis_fetch.py')
CORE_SCRIPT   = os.path.join(ORBIS_ROOT, 'codebase', 'orbis_core.py')
CACHE_DIR     = '/tmp/orbis_cache'

os.makedirs(OUT_DIR,   exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# ── 1.2 Installer requests si absent ─────────────────────
try:
    import requests
    print('[SETUP] requests : OK')
except ImportError:
    subprocess.run(['pip', 'install', 'requests', '-q'])
    print('[SETUP] requests : installe')

# ── 1.3 Installer Blender 4.2 LTS ────────────────────────
BLENDER_URL  = 'https://download.blender.org/release/Blender4.2/blender-4.2.0-linux-x64.tar.xz'
BLENDER_DIR  = '/content/blender'
BLENDER_BIN  = os.path.join(BLENDER_DIR, 'blender')

# Chercher d'abord dans le cache Drive
BLENDER_DRIVE_CACHE = os.path.join(DRIVE_ROOT, 'tools', 'blender-4.2.0-linux-x64', 'blender')

if os.path.isfile(BLENDER_DRIVE_CACHE):
    BLENDER_BIN = BLENDER_DRIVE_CACHE
    print(f'[SETUP] Blender (cache Drive) : {BLENDER_BIN}')
elif os.path.isfile(BLENDER_BIN):
    print(f'[SETUP] Blender deja installe : {BLENDER_BIN}')
else:
    print('[SETUP] Telechargement Blender 4.2 LTS...')
    subprocess.run(['wget', '-q', '--show-progress', BLENDER_URL, '-O', '/tmp/blender.tar.xz'])
    subprocess.run(['mkdir', '-p', BLENDER_DIR])
    subprocess.run(['tar', '-xf', '/tmp/blender.tar.xz', '-C', BLENDER_DIR, '--strip-components=1'])
    print(f'[SETUP] Blender installe : {BLENDER_BIN}')

# Verification
result = subprocess.run([BLENDER_BIN, '--version'], capture_output=True, text=True)
print(f'[SETUP] {result.stdout.splitlines()[0] if result.stdout else "version inconnue"}')

# ── 1.4 Bilan ────────────────────────────────────────────
print()
print('[SETUP] ══════════════════════════════════════')
scripts_ok = all(os.path.isfile(p) for p in [FETCH_SCRIPT, CORE_SCRIPT])
print(f'  Scripts   : {"OK" if scripts_ok else "MANQUANTS — executer GIT SYNC"}')
print(f'  Blender   : {"OK" if os.path.isfile(BLENDER_BIN) else "MANQUANT"}')
print(f'  IN/       : {IN_DIR}')
print(f'  OUT/      : {OUT_DIR}')
print(f'  Cache     : {CACHE_DIR}')
print('[SETUP] Passer a la cellule CONFIG ▶')
print('[SETUP] ══════════════════════════════════════')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 2 — CONFIG
# Definir ici : ASSET_ID ou KEYWORD
# ═══════════════════════════════════════════════════════════

import os

# ── CONFIGURER ICI ─────────────────────────────────────────
# Choisir UN des deux modes :
#   MODE = "ASSET_ID"  → renseigner ASSET_ID (deterministe, recommande)
#   MODE = "KEYWORD"   → renseigner KEYWORD  (exploration)

MODE     = "ASSET_ID"              # ou "KEYWORD"
ASSET_ID = ""                      # ex: "130829806"  (Brookhaven)
KEYWORD  = "brookhaven street"     # utilise seulement si MODE = KEYWORD

# ── Detection auto depuis IN/ ─────────────────────────────
# Alternative : deposer un fichier asset_id.txt ou keyword.txt dans IN/
asset_id_file = os.path.join(IN_DIR, 'asset_id.txt')
keyword_file  = os.path.join(IN_DIR, 'keyword.txt')

if os.path.isfile(asset_id_file):
    with open(asset_id_file) as f:
        ASSET_ID = f.read().strip()
    MODE = "ASSET_ID"
    print(f'[CONFIG] asset_id.txt detecte : {ASSET_ID}')
elif os.path.isfile(keyword_file):
    with open(keyword_file) as f:
        KEYWORD = f.read().strip()
    MODE = "KEYWORD"
    print(f'[CONFIG] keyword.txt detecte : {KEYWORD}')

# ── Validation ────────────────────────────────────────────
ready = False
if MODE == "ASSET_ID" and ASSET_ID:
    print(f'[CONFIG] Mode : ASSET_ID = {ASSET_ID}')
    ready = True
elif MODE == "KEYWORD" and KEYWORD:
    print(f'[CONFIG] Mode : KEYWORD = "{KEYWORD}"')
    ready = True
else:
    print('[CONFIG] ERREUR — Renseigner ASSET_ID ou KEYWORD ci-dessus')

# ── Nom du cache ──────────────────────────────────────────
CACHE_SUBDIR = os.path.join(CACHE_DIR, ASSET_ID if ASSET_ID else 'keyword_search')
os.makedirs(CACHE_SUBDIR, exist_ok=True)

print()
print('[CONFIG] ══════════════════════════════════════')
print(f'  Mode     : {MODE}')
print(f'  Input    : {ASSET_ID if MODE == "ASSET_ID" else KEYWORD}')
print(f'  Cache    : {CACHE_SUBDIR}')
print(f'  GO       : {"OUI" if ready else "NON — corriger config ci-dessus"}')
print('[CONFIG] ══════════════════════════════════════')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 3 — EXTRACTION HTTP
# Lance orbis_fetch.py : API Roblox + CDN textures → metadata.json
# ═══════════════════════════════════════════════════════════

import subprocess, time, json, os

if not ready:
    print('[FETCH] Conditions non remplies — executer CONFIG dabord')
    raise SystemExit

print('[FETCH] ══════════════════════════════════════')
print('[FETCH] Lancement orbis_fetch.py...')
t0 = time.time()

# Construire la commande selon le mode
fetch_cmd = ['python', FETCH_SCRIPT, '--out-dir', CACHE_SUBDIR]

if MODE == "ASSET_ID":
    fetch_cmd += ['--asset-id', ASSET_ID]
else:
    fetch_cmd += ['--keyword', KEYWORD]

proc = subprocess.run(fetch_cmd, capture_output=True, text=True)
elapsed = round(time.time() - t0, 1)

# Afficher les lignes ORBIS
orbis_lines = [l for l in proc.stdout.splitlines() if '[ORBIS]' in l]
for line in orbis_lines:
    print(line)

if proc.returncode != 0:
    print(f'[FETCH] ERREUR (code {proc.returncode})')
    for line in proc.stderr.splitlines()[-10:]:
        print(f'  {line}')
    raise RuntimeError('orbis_fetch.py echoue')

# Lire metadata.json produit
METADATA_PATH = os.path.join(CACHE_SUBDIR, 'metadata.json')
if not os.path.isfile(METADATA_PATH):
    raise RuntimeError(f'metadata.json absent : {METADATA_PATH}')

with open(METADATA_PATH) as f:
    meta = json.load(f)

# Si mode KEYWORD, recuperer l'asset_id resolu
if MODE == 'KEYWORD' and not ASSET_ID:
    ASSET_ID = meta.get('asset_id', 'unknown')

print()
print(f'[FETCH] TERMINE en {elapsed}s')
print(f'  Asset ID  : {meta.get("asset_id")}')
print(f'  Format    : {meta.get("format")}')
print(f'  Parts     : {meta.get("parts_count")}')
print(f'  Textures  : {meta["textures"]["resolues"]}/{meta["textures"]["total"]} resolues')
if meta.get('warnings'):
    for w in meta['warnings']:
        print(f'  WARNING   : {w}')
print('[FETCH] ══════════════════════════════════════')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 4 — PIPELINE BLENDER
# Lance orbis_core.py headless : metadata.json → GLB
# Recycle locus_main.ipynb Cell 04
# ═══════════════════════════════════════════════════════════

import subprocess, time, os

OUTPUT_GLB = os.path.join(OUT_DIR, f'decor_{ASSET_ID}.glb')

print('[PIPELINE] ══════════════════════════════════════')
print(f'[PIPELINE] Asset    : {ASSET_ID}')
print(f'[PIPELINE] Metadata : {METADATA_PATH}')
print(f'[PIPELINE] Output   : {OUTPUT_GLB}')
print('[PIPELINE] Lancement Blender headless...')

t0  = time.time()
cmd = [
    BLENDER_BIN, '--background',
    '--python', CORE_SCRIPT,
    '--',
    '--metadata', METADATA_PATH,
    '--asset-id', str(ASSET_ID),
    '--output',   OUTPUT_GLB,
]

result  = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
elapsed = round(time.time() - t0, 1)

# Afficher les lignes ORBIS du stdout
orbis_lines = [l for l in result.stdout.splitlines() if '[ORBIS]' in l]
for line in orbis_lines:
    print(line)

if result.returncode != 0:
    print(f'[PIPELINE] ORBIS ECHEC (code {result.returncode})')
    print('[PIPELINE] Derniers messages Blender :')
    for line in result.stderr.splitlines()[-15:]:
        print(f'  {line}')
    raise RuntimeError(f'orbis_core.py echoue (code {result.returncode})')

print()
print(f'[PIPELINE] COMPLETE en {elapsed}s')
if os.path.isfile(OUTPUT_GLB):
    size_mb = os.path.getsize(OUTPUT_GLB) / (1024*1024)
    print(f'[PIPELINE] GLB     : {OUTPUT_GLB} ({size_mb:.2f} Mo)')
else:
    print('[PIPELINE] WARNING — GLB non trouve en sortie')
print('[PIPELINE] ══════════════════════════════════════')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 5 — RAPPORT
# Lit rapport_orbis.json et affiche le bilan
# Recycle locus_main.ipynb Cell 05
# ═══════════════════════════════════════════════════════════

import json, os

RAPPORT_PATH = os.path.join(OUT_DIR, 'rapport_orbis.json')

if not os.path.isfile(RAPPORT_PATH):
    print('[RAPPORT] rapport_orbis.json introuvable — executer PIPELINE dabord')
else:
    with open(RAPPORT_PATH) as f:
        r = json.load(f)

    status = r.get('status', '?')
    icon   = '✓' if status == 'SUCCESS' else '✗'

    print('[RAPPORT] ══════════════════════════════════════')
    print(f'[RAPPORT]  {icon} STATUS     : {status}')
    print(f'[RAPPORT]    Asset ID   : {r.get("asset_id")}')
    print(f'[RAPPORT]    Format     : {r.get("format")}')

    mesh = r.get('mesh', {})
    print(f'[RAPPORT]    Parts      : {mesh.get("parts_roblox", "?")}')
    print(f'[RAPPORT]    Vertices   : {mesh.get("vertices", "?"):,}' if isinstance(mesh.get('vertices'), int) else f'[RAPPORT]    Vertices   : ?')
    print(f'[RAPPORT]    Faces      : {mesh.get("faces", "?"):,}' if isinstance(mesh.get('faces'), int) else f'[RAPPORT]    Faces      : ?')

    tex = r.get('textures', {})
    print(f'[RAPPORT]    Textures   : {tex.get("resolues",0)}/{tex.get("total",0)} resolues ({tex.get("privees",0)} privees)')
    print(f'[RAPPORT]    Double face: {"OUI" if r.get("double_face") else "NON"}')
    print(f'[RAPPORT]    GLB size   : {r.get("output_size_mb", "?"):.2f} Mo' if isinstance(r.get('output_size_mb'), (int, float)) else f'[RAPPORT]    GLB size   : ?')

    if r.get('warnings'):
        print()
        for w in r['warnings']:
            print(f'[RAPPORT]    WARNING    : {w}')

    if r.get('error'):
        print(f'[RAPPORT]    ERREUR     : {r["error"]}')

    print()
    if status == 'SUCCESS':
        print(f'[RAPPORT] GLB pret : {r.get("output_glb")}')
        print('[RAPPORT] Executer la cellule DOWNLOAD ▶')
    else:
        print('[RAPPORT] Pipeline en erreur — verifier les messages ci-dessus')
    print('[RAPPORT] ══════════════════════════════════════')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 6 — DOWNLOAD
# Telecharge le GLB et le rapport JSON
# Recycle locus_main.ipynb Cell 06
# ═══════════════════════════════════════════════════════════

import os
from google.colab import files

OUTPUT_GLB   = os.path.join(OUT_DIR, f'decor_{ASSET_ID}.glb')
RAPPORT_PATH = os.path.join(OUT_DIR, 'rapport_orbis.json')

print('[DOWNLOAD] Telechargement des fichiers...')

if os.path.isfile(OUTPUT_GLB):
    size_mb = os.path.getsize(OUTPUT_GLB) / (1024*1024)
    print(f'[DOWNLOAD] GLB    : {os.path.basename(OUTPUT_GLB)} ({size_mb:.2f} Mo)')
    files.download(OUTPUT_GLB)
else:
    print(f'[DOWNLOAD] GLB introuvable : {OUTPUT_GLB}')

if os.path.isfile(RAPPORT_PATH):
    print(f'[DOWNLOAD] Rapport : rapport_orbis.json')
    files.download(RAPPORT_PATH)
else:
    print(f'[DOWNLOAD] Rapport introuvable : {RAPPORT_PATH}')

print('[DOWNLOAD] POUR L\'EMPEROR. POUR LA FLOTTE FERRUS.')